In [0]:
from pyspark.sql import SparkSession
import pyspark.sql.functions

#### Reading files

In [0]:
spark = SparkSession \
    .builder \
    .appName("Python Spark SQL basic example") \
    .config("spark.some.config.option", "some-value") \
    .getOrCreate()  # initializing session

In [0]:
my_volume_dir = "/Volumes/dbr_dev/lechster10/databricks_course/first_week_resources"

path_game = f"{my_volume_dir}/game.json"
path_game_summ = f"{my_volume_dir}/game_summary.csv"
path_team = f"{my_volume_dir}/team.csv"

df_game = spark.read.option("multiLine", "true").format("json").load(path_game)
df_game_summ = spark.read.format("csv").option("header","true").load(path_game_summ)
df_team = spark.read.format("csv").option("header","true").load(path_team)


#### Loading into delta table

In [0]:
catalog = "dbr_dev"
schema = "lechster10"

df_game.write.format("delta").saveAsTable(f"{catalog}.{schema}.game_table")
df_game_summ.write.format("delta").saveAsTable(f"{catalog}.{schema}.df_game_summ_table")
df_team.write.format("delta").saveAsTable(f"{catalog}.{schema}.df_team_table")


#### Simple operations

In [0]:
df_game.show(3)

In [0]:
df_game_summ.show(3)

In [0]:
df_team.show(3)

In [0]:
old_column = "full_name"

results_df = df_game_summ.join(df_game, "game_id")
results_with_team_names = results_df.join(df_team, results_df["home_team_id"] == df_team["id"])   # adding home team name
results_with_team_names = results_with_team_names.select(["game_id", "game_date_est", "visitor_team_id", "season", "pts_home", "pts_away", old_column]) 
results_with_team_names = results_with_team_names.withColumnRenamed('full_name', 'home_team_name')

results_with_team_names = results_with_team_names.join(df_team, results_df["visitor_team_id"] == df_team["id"])   # adding visitor team name
results_with_team_names = results_with_team_names.select(["game_id", "game_date_est", "visitor_team_id", "season", "pts_home", "pts_away", "home_team_name", old_column]) 
results_with_team_names = results_with_team_names.withColumnRenamed('full_name', 'visitor_team_name')

results_with_team_names.show()
